# Generate Few Shot Examples from Train Data
This notebook extracts 2 examples per label (0-8) from the training data, formatted exactly how it will be passed to Gemini. You can copy the outputs from here directly into your `few_shot_examples.txt` file and add the expected reasoning/prediction.

In [ ]:
import json
import random
from collections import defaultdict

label_map = {
    0: "No Defense / Neutral Utterance",
    1: "Action Defense Level",
    2: "Major Image-distorting Defense Level",
    3: "Disavowal Defense Level",
    4: "Minor Image-distorting Defense Level",
    5: "Neurotic Defense Level",
    6: "Obsessional Defense Level",
    7: "Highly Adaptive Defense Level",
    8: "Need More Information"
}

with open('input_data/train.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

examples_by_label = defaultdict(list)
for item in data:
    if 'label' in item:
        examples_by_label[item['label']].append(item)

print(f"Total items loaded: {len(data)}")
print(f"Labels found: {sorted(examples_by_label.keys())}")


for label in sorted(examples_by_label.keys()):
    label_name = label_map.get(label, "Unknown")
    print(f"\n=======================================================")
    print(f"                    LABEL {label}: {label_name}")
    print(f"=======================================================")
    
    # Get up to 2 random examples for this label
    samples = random.sample(examples_by_label[label], min(2, len(examples_by_label[label])))
    for i, item in enumerate(samples):
        print(f"\n--- Example {i+1} ---")
        
        # Exact prompt logic from gemini_predict.py
        dialogue_lines = []
        for turn in item.get('dialogue', []):
            speaker = turn.get('speaker', '').capitalize()
            text = turn.get('text', '')
            dialogue_lines.append(f"{speaker}: {text}")
            
        formatted_dialogue = "\n".join(dialogue_lines)
        user_prompt = f"{formatted_dialogue}\n\ncurrent_text_to_classify: {item.get('current_text', '')}"
        
        print(user_prompt)
        print("\n[YOUR JSON REASONING AND CLASSIFICATION HERE]")
        print("{")
        print('  "clinical_reasoning": {')
        print('    "context_trigger": "...",')
        print('    "psychological_goal": "...",')
        print('    "handbook_alignment": "...",')
        print('    "differential_diagnosis": "..."')
        print('  },')
        print(f'  "defense_level": {label},')
        print(f'  "label": "{label_name}"')
        print("}\n")